# Multiple dispatch vs POO Python · **côté Python**

*Piste Python / Colab — analogue de `2_dispatch_vs_poo.jl`.*

> **À lire d'abord.** Le **dispatch multiple ne sert quasiment à rien en Python** : ce n'est pas
> dans le langage, ce n'est pas idiomatique, et **Python n'est pas fait pour ça** — sa philosophie,
> c'est le *duck typing* (on appelle la méthode, on ne dispatche pas sur les types). Si vous voulez
> le dispatch multiple comme **outil de travail**, c'est **Julia**, pas Python.
>
> Le but de ce notebook n'est donc **pas** de vous apprendre à coder ainsi en Python, mais de
> **comprendre le contraste** : ce que Julia offre **nativement** (et compilé → rapide), Python ne
> peut que l'**émuler** avec une bibliothèque tierce (et à l'exécution → sans gain de vitesse).

⚠️ Module **conceptuel** (un point de langage), pas un benchmark : on illustre ce que la POO de Python
fait — et ce qu'elle ne fait **pas**.

## 1. La POO : la méthode vit *dans* l'objet

Le choix de la méthode se fait sur **un seul** objet — celui à gauche du point (`obj.aire()`).

In [ ]:
import math

class Forme: ...

class Cercle(Forme):
    def __init__(self, r): self.r = r
    def aire(self): return math.pi * self.r**2

class Rectangle(Forme):
    def __init__(self, L, l): self.L, self.l = L, l
    def aire(self): return self.L * self.l

formes = [Cercle(2.0), Rectangle(3.0, 4.0), Cercle(1.0)]
[f.aire() for f in formes]

**Ajouter une opération** (disons `perimetre`) oblige à **modifier chaque classe** — ou à écrire
une fonction externe qui teste les types à la main (`isinstance`), ce qui est lourd et fermé.

In [ ]:
def perimetre(f):
    if isinstance(f, Cercle):
        return 2*math.pi*f.r
    elif isinstance(f, Rectangle):
        return 2*(f.L + f.l)
    raise TypeError(f"perimetre non défini pour {type(f).__name__}")

[perimetre(f) for f in formes]

## 2. Le dispatch « simple » de Python : `singledispatch`

`functools.singledispatch` choisit la méthode selon le type du **premier** argument seulement —
un dispatch *simple*, pas *multiple*.

In [ ]:
from functools import singledispatch

@singledispatch
def aire(forme): raise NotImplementedError

@aire.register
def _(c: Cercle): return math.pi * c.r**2

@aire.register
def _(r: Rectangle): return r.L * r.l

[aire(f) for f in formes]

## 3. Là où Python coince : le dispatch sur **plusieurs** arguments

Imaginons `interaction(a, b)` : ce qu'on renvoie dépend du **couple** de types —
`(Cercle, Cercle)`, `(Cercle, Rectangle)`, `(Rectangle, Rectangle)`.

En POO on écrirait `a.interaction(b)`. Mais Python trouve la méthode en regardant **seulement
`type(a)`** (l'objet à gauche du point) ; `type(b)` n'entre **pas** dans le choix de la méthode. On
est donc forcé de tester `b` à la main, *à l'intérieur* :

```python
class Cercle:
    def interaction(self, b):
        if   isinstance(b, Cercle):    return "deux cercles"
        elif isinstance(b, Rectangle): return "cercle et rectangle"
        ...
```

→ des cascades de `isinstance`, et il faut **rouvrir chaque classe** dès qu'un type apparaît.

⚠️ **Python n'a PAS de dispatch multiple.** La bibliothèque standard n'offre que `singledispatch`
(un seul argument). Pour le dispatch multiple il faut une **bibliothèque tierce**, `multipledispatch`
(`pip install`) — ce n'est pas le langage. On la montre ici **pour illustrer** ce que Julia fait
nativement : on enregistre une implémentation **par combinaison de types** (`@dispatch(Cercle, Rectangle)`),
et à l'appel elle inspecte le type **des deux** arguments :

In [ ]:
try:
    from multipledispatch import dispatch
except ImportError:
    !pip -q install multipledispatch
    from multipledispatch import dispatch

@dispatch(Cercle, Cercle)
def interaction(a, b): return "deux cercles"

@dispatch(Cercle, Rectangle)
def interaction(a, b): return "cercle et rectangle"

@dispatch(Rectangle, Rectangle)
def interaction(a, b): return "deux rectangles"

[interaction(Cercle(1.0), Cercle(2.0)),
 interaction(Cercle(1.0), Rectangle(2.0, 3.0)),
 interaction(Rectangle(1.,1.), Rectangle(2.,2.))]

**Donc : Python propose-t-il le dispatch multiple ? Non — et à quoi sert cette lib ?**

- Ce n'est **pas** dans le langage : `multipledispatch` est une dépendance externe, **peu utilisée**
  (la culture Python = duck typing : on appelle la méthode, on ne dispatche pas sur les types).
- Elle rend service dans des cas **niches** : opérations binaires entre types maison, ou pour éviter
  des cascades de `isinstance` (façon *pattern visiteur*).
- **Mais elle n'apporte pas la vitesse de Julia.** Elle fait sa recherche **à l'exécution** (types
  découverts au runtime → aucune spécialisation). Julia fait *dispatch sur les types → **compilation**
  native*. Même fonctionnalité apparente, mécanisme — et performance — **opposés**.

> On la montre donc pour **mesurer l'écart** : *natif + compilé* (Julia, rapide) vs *bibliothèque
> + runtime* (Python, pratique mais pas plus rapide). Encore la distinction **exécution vs compilation**.

## 4. Au fait : et le *duck typing* ?

On pourrait croire que « Python ne dispatche pas vraiment, parce qu'il est *duck typed* ». À préciser :

- **L'absence de dispatch multiple** ne vient PAS du duck typing, mais du **dispatch simple** du
  modèle objet : `a.method(b)` ne consulte que `type(a)`. Même un Python à types stricts l'aurait.
- **Le duck typing joue ailleurs.** En Python idiomatique, on **évite** de dispatcher sur les types :
  on appelle la méthode et on fait confiance à l'objet (« s'il cancane comme un canard… », style EAFP).
  Dispatcher explicitement sur les types (ci-dessus) va à *rebours* du style Python.
- **Lien avec le module 1 (la vitesse).** Comme Python est dynamiquement / duck typed, le type de
  chaque objet n'est connu qu'**à l'exécution**. Donc quand dispatch il y a, il se fait à l'exécution,
  par recherche, **sans spécialisation** — re-payé à chaque appel. Julia fait l'inverse : les types
  sont connus **à la compilation** (JIT) → il choisit la méthode *et* compile une version native
  spécialisée, une fois. Le **même** mécanisme assure la **correction** (choisir la bonne méthode)
  **et** la **vitesse**.

> **Distinction clé, côté dispatch.** Type **découvert à l'exécution** (Python → re-vérifié
> à chaque opération, lent) vs type **connu à la compilation** (Julia → spécialisé une fois, rapide).

## Bilan

| | Python (POO) | Julia (dispatch) |
|---|---|---|
| Où vit la méthode | **dans** l'objet | **hors** des types, fonction générique |
| Choisie selon | le type de `self` (un seul) | le type de **tous** les arguments |
| Multi-arguments | bibliothèque tierce (`multipledispatch`) | **natif** |
| Typage | duck / dynamique → dispatch runtime, pas de spécialisation | types → **compilation spécialisée** |
| Lien avec la vitesse | — | **le moteur** de la spécialisation/compilation |

> En Julia le dispatch multiple est *natif* et **génère du code natif spécialisé** : c'est exactement
> ce qui le rend rapide. En Python, c'est un patron de conception ajouté, sans gain de performance.